In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    roc_curve
)

pd.set_option("display.max_columns", 100)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

train = pd.read_csv(DATA / "train.csv")

TARGET = "is_fraud"
ID_COL = "id"

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].astype(int)

print("Features:", X.shape)
print("Target:", y.shape)

Features: (699960, 24)
Target: (699960,)


#### Create Engineered Features

In [3]:
def create_features(df):
    df = df.copy()

    # Transaction frequency relative to elapsed time
    df["txn_velocity"] = (
        df["txn_count_last_24h"] /
        (df["hours_since_last_txn"] + 1)
    )

    # Transaction amount relative to account balance
    df["amount_balance_ratio"] = (
        df["amount_usd"] /
        (df["account_balance_usd"].abs() + 1)
    )

    # Interaction between transaction amount and merchant risk
    df["amount_merchant_risk"] = (
        df["amount_usd"] * df["merchant_risk_score"]
    )

    # Interaction between transaction distance and amount
    df["distance_amount_interaction"] = (
        df["distance_from_home_km"] * df["amount_usd"]
    )

    return df

#### Generate the engineered dataset

In [4]:
X_fe = create_features(X)

new_features = [
    "txn_velocity",
    "amount_balance_ratio",
    "amount_merchant_risk",
    "distance_amount_interaction"
]

display(X_fe[new_features].describe().round(3))

,txn_velocity,amount_balance_ratio,amount_merchant_risk,distance_amount_interaction
count,699960.000,699960.000,699960.000,699960.000
mean,0.731,0.067,3251.008,1090.456
std,0.762,0.205,9476.748,3810.574
min,0.000,0.000,0.000,0.000
25%,0.232,0.009,475.590,94.145
50%,0.473,0.019,865.242,261.327
75%,0.942,0.040,1510.555,646.523
max,9.709,14.365,416604.813,264161.898


In [5]:
print("Missing values:")
display(X_fe[new_features].isnull().sum())

print("Infinite values:")
display(
    pd.Series(
        np.isinf(X_fe[new_features]).sum(),
        index=new_features
    )
)

Missing values:


txn_velocity                   0
amount_balance_ratio           0
amount_merchant_risk           0
distance_amount_interaction    0
dtype: int64

Infinite values:


txn_velocity                   0
amount_balance_ratio           0
amount_merchant_risk           0
distance_amount_interaction    0
dtype: int64

#### Train-Validation Split 

In [9]:
X_train_fe, X_val_fe, y_train_fe, y_val_fe = train_test_split(
    X_fe,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [10]:
print("Validation rows:", X_val_fe.shape[0])
print("Fraud cases:", y_val_fe.sum())
print("Fraud rate:", round(y_val_fe.mean(), 5))

Validation rows: 139992
Fraud cases: 2338
Fraud rate: 0.0167


#### Build preprocessing pipeline

In [11]:
num_cols_fe = X_fe.select_dtypes(include=np.number).columns.tolist()

cat_cols_fe = X_fe.select_dtypes(
    include=["object", "str", "bool"]
).columns.tolist()

print("Numerical features:", len(num_cols_fe))
print("Categorical features:", len(cat_cols_fe))

Numerical features: 17
Categorical features: 11


In [13]:
preprocessor_fe = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        num_cols_fe
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols_fe
    )
])

#### Train LightGBM

In [15]:
lgb_fe = Pipeline([
    ("preprocessor", preprocessor_fe),
    ("model", lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

lgb_fe.fit(X_train_fe, y_train_fe)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](28,)","['amount_usd','hours_since_last_txn','txn_count_last_24h',..., 'amount_balance_ratio','amount_merchant_risk', 'distance_amount_interaction']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,28
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (d

#### Generate Predictions

In [16]:
fe_probs = lgb_fe.predict_proba(X_val_fe)[:, 1]

print("Predictions generated:", len(fe_probs))

Predictions generated: 139992


#### Evaluate the engineered model

In [17]:
fe_pr_auc = average_precision_score(y_val_fe, fe_probs)
fe_roc_auc = roc_auc_score(y_val_fe, fe_probs)

fpr, tpr, thresholds = roc_curve(y_val_fe, fe_probs)

valid = np.where(fpr <= 0.01)[0]
idx = valid[np.argmax(tpr[valid])]

fe_recall_1pct = tpr[idx]
fe_threshold_1pct = thresholds[idx]

print("PR-AUC:", round(fe_pr_auc, 5))
print("ROC-AUC:", round(fe_roc_auc, 5))
print("Recall at 1% FPR:", round(fe_recall_1pct, 5))
print("Threshold at 1% FPR:", round(fe_threshold_1pct, 5))

PR-AUC: 0.49902
ROC-AUC: 0.94362
Recall at 1% FPR: 0.50342
Threshold at 1% FPR: 0.18981
